In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName('08').getOrCreate()

In [3]:
zomato = spark.read.csv('C:/Users/User/Downloads/zomato.csv',header=True,inferSchema=True)

In [4]:
zomato.show(5)

+--------------------+--------------------+--------------------+--------------------+--------------------+-----+--------------------+--------------+--------------------+-------------------+--------------------+--------------------+---------------------------+--------------------+--------------------+--------------------+--------------------+
|                 url|             address|                name|        online_order|          book_table| rate|               votes|         phone|            location|          rest_type|          dish_liked|            cuisines|approx_cost(for two people)|        reviews_list|           menu_item|     listed_in(type)|     listed_in(city)|
+--------------------+--------------------+--------------------+--------------------+--------------------+-----+--------------------+--------------+--------------------+-------------------+--------------------+--------------------+---------------------------+--------------------+--------------------+-----------

In [5]:
zomato.printSchema()

root
 |-- url: string (nullable = true)
 |-- address: string (nullable = true)
 |-- name: string (nullable = true)
 |-- online_order: string (nullable = true)
 |-- book_table: string (nullable = true)
 |-- rate: string (nullable = true)
 |-- votes: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- location: string (nullable = true)
 |-- rest_type: string (nullable = true)
 |-- dish_liked: string (nullable = true)
 |-- cuisines: string (nullable = true)
 |-- approx_cost(for two people): string (nullable = true)
 |-- reviews_list: string (nullable = true)
 |-- menu_item: string (nullable = true)
 |-- listed_in(type): string (nullable = true)
 |-- listed_in(city): string (nullable = true)



In [6]:
from pyspark.sql.types import IntegerType
df = zomato.\
withColumn('approx_cost(for two people)',zomato['approx_cost(for two people)'].\
           cast(IntegerType())).\
withColumn('votes',zomato['votes'].cast(IntegerType()))

In [7]:
df.printSchema()

root
 |-- url: string (nullable = true)
 |-- address: string (nullable = true)
 |-- name: string (nullable = true)
 |-- online_order: string (nullable = true)
 |-- book_table: string (nullable = true)
 |-- rate: string (nullable = true)
 |-- votes: integer (nullable = true)
 |-- phone: string (nullable = true)
 |-- location: string (nullable = true)
 |-- rest_type: string (nullable = true)
 |-- dish_liked: string (nullable = true)
 |-- cuisines: string (nullable = true)
 |-- approx_cost(for two people): integer (nullable = true)
 |-- reviews_list: string (nullable = true)
 |-- menu_item: string (nullable = true)
 |-- listed_in(type): string (nullable = true)
 |-- listed_in(city): string (nullable = true)



## 1. `NULL` data 수 확인하기

- `col.isNull()`: column의 값이 Null이면 True, Null이 아니면 False

In [8]:
print(df.columns)

['url', 'address', 'name', 'online_order', 'book_table', 'rate', 'votes', 'phone', 'location', 'rest_type', 'dish_liked', 'cuisines', 'approx_cost(for two people)', 'reviews_list', 'menu_item', 'listed_in(type)', 'listed_in(city)']


In [9]:
df['book_table'].isNull()

Column<'(book_table IS NULL)'>

- `DataFrame.where(DataFrame[col].isNull())`: col의 값이 Null인 DataFrame의 행(Row)만 반환

In [10]:
df.where(df.book_table.isNull()).show(5)
# book_table 열의 값이 NULL인 행만 반환

+---------------+------------+-----------+------------+----------+----+-----+-----+--------+------------+----------+--------+---------------------------+------------+---------+---------------+---------------+
|            url|     address|       name|online_order|book_table|rate|votes|phone|location|   rest_type|dish_liked|cuisines|approx_cost(for two people)|reviews_list|menu_item|listed_in(type)|listed_in(city)|
+---------------+------------+-----------+------------+----------+----+-----+-----+--------+------------+----------+--------+---------------------------+------------+---------+---------------+---------------+
|+91 9945435158"|Banashankari|Quick Bites|        NULL|      NULL| 150| NULL|   []|Delivery|Banashankari|      NULL|    NULL|                       NULL|        NULL|     NULL|           NULL|           NULL|
|+91 9945435158"|Banashankari|Quick Bites|        NULL|      NULL| 150| NULL|   []|Dine-out|Banashankari|      NULL|    NULL|                       NULL|        NUL

- `DataFrame.where(DataFrame[col].isNull()).count()`: col값이 Null인 DataFrame의 행(Row) 수를 반환

In [11]:
df.where(df.book_table.isNull()).count()\
# df.where(df['book_table'].isNull()).count()
# book_table 열의 값이 Null인 행은 총 2개이다.

2

- **mission**: 각 column에 대해, Null인 데이터의 수 출력

In [13]:
for col in df.columns:
    print(col, df.where(df[col].isNull()).count())

url 0
address 0
name 85
online_order 8111
book_table 2
rate 7775
votes 20018
phone 1227
location 20054
rest_type 20165
dish_liked 46841
cuisines 27305
approx_cost(for two people) 43611
reviews_list 28185
menu_item 28611
listed_in(type) 28983
listed_in(city) 29344


## 2. **missing value(`NULL`)** 를 포함한 행(`Row`) 삭제하기

- `DataFrame.na.drop(how,thresh,subset)`
- `how`를 **설정하지 않는** 경우, **하나의 column이라도 NULL을 포함하고 있다면 해당 행을 삭제**함

In [14]:
# 전체 행의 수
df.count()

71730

In [15]:
# NULL 하나이상 포함된 행을 삭제한 결과
df.na.drop().count()

8606

In [16]:
# 삭제되고 남은 데이터 행의 비율
df.na.drop().count()/df.count()*100

11.997769413076815

- `how = 'all'`: 모든 column의 값이 NULL인 경우, 해당 행 삭제

In [17]:
df.na.drop(how='all').count()/df.count()*100
# 결과가 100.0이면, 삭제된 행이 없다는 의미
# 즉, 모든 열의 값이 NULL인 행은 없음

100.0

- `DataFrame.na.drop(thresh)`: `thresh`로 설정된 수보다 **NULL이 아닌 데이터의 수가 적은 경우**, 행을 삭제
> - `DataFrame.na.drop(thresh=5)`: NULL이 아닌 값이 **5개 미만이면 행을 삭제**, 각 행에 NULL이 아닌 값이 최소 5개 이상 있어야 한다는 의미

In [18]:
# DataFrame df column의 수
len(df.columns)

17

In [19]:
# 4개 이상의 NULL이 있는 행을 삭제
# NULL이 3개 미만이면 삭제하지 않음
# df는 총 17개의 열을 가지고 있으므로, NULL이 아닌 열의 값이 14개 미만이면 행을 삭제
print(df.na.drop(thresh=14).count())
print(df.na.drop(thresh=14).count()/df.count()*100)

42391
59.09800641293741


- `DataFrame.na.drop(subset=col)`: 지정된 column의 값이 NULL인 경우, 행 삭제

In [20]:
print(df.count()) # 71730
print(df.na.drop(subset='book_table').count()) # 71728
# 값이 NULL인 book_table 열은 2개이므로, 2개의 행이 삭제됨

71730
71728


- `DataFrame.na.drop(subset=[col1,col2])`: 2개 이상의 column을 지정하는 경우, **하나라도 NULL이 포함되어 있으면 해당 행을 삭제**

In [21]:
print(df.na.drop(subset='rate').count())
print(df.na.drop(subset='phone').count())

63955
70503


In [22]:
print(df.na.drop(subset=['rate','phone']).count())
# 만약 rate와 phone 둘다 NULL인 행만 삭제된다면, 위 결과는 70503보다 커야 하고
# 만약 rate와 phone 둘 중 하나라도 NULL인 경우 삭제된다면, 위 결과는 63955보다 작아야 함

63104


In [23]:
# 삭제된 행의 수 비교
print('rate NULL : ', df.count() - df.na.drop(subset='rate').count())
print('phone NULL: ', df.count() - df.na.drop(subset='phone').count())
print('rate OR phone NULL: ', df.count() - df.na.drop(subset=['rate','phone']).count())

rate NULL :  7775
phone NULL:  1227
rate OR phone NULL:  8626


- `filter`를 사용하여 특정 column의 값이 NULL인 행을 삭제할 수 있음
- `DataFrame.filter(True/False)`: True인 행만 필터링

- `col.isNotNull()`: column의 값이 NULL이 아닌 경우 `True`

In [25]:
# rate 열의 값이 존재하는(NULL이 아닌) 행만 반환
df.filter(df.rate.isNotNull()).count()

63955

In [26]:
# phone 열의 값이 존재하는(NULL이 아닌) 행만 반환
df.filter(df.phone.isNotNull()).count()

70503

In [27]:
# filter를 사용해서 rate와 phone 둘 중 하나라도 Null인 행 삭제
# 1. rate가 Null이 아닌 행만 필터링 (rate가 Null인 행 삭제)
# 2. 1번 결과에서  phone이 Null이 아닌 행만 필터링 (rate 또는 phone이 Null인 행 삭제)
df.filter(df.rate.isNotNull()).filter(df.phone.isNotNull()).count()

63104

In [28]:
# rate가 Null이 아닌 행의 수
print(df.filter(df.rate.isNotNull()).count())
# rate 열이 NULL인 행의 수
print(df.filter(df.rate.isNull()).count())
# votes 열이 NULL인 행의 수 + votes 열의 값이 존재하는(NULL이 아닌) 행의 수 = 전체 행의 수

63955
7775


## 3. `NULL` **대체**하기

- `DataFrame.na.fill()`

In [29]:
df.show(3)

+--------------------+--------------------+--------------+--------------------+--------------------+-----+-----+-------------+--------------------+-------------+--------------------+--------------------+---------------------------+--------------------+--------------------+---------------+--------------------+
|                 url|             address|          name|        online_order|          book_table| rate|votes|        phone|            location|    rest_type|          dish_liked|            cuisines|approx_cost(for two people)|        reviews_list|           menu_item|listed_in(type)|     listed_in(city)|
+--------------------+--------------------+--------------+--------------------+--------------------+-----+-----+-------------+--------------------+-------------+--------------------+--------------------+---------------------------+--------------------+--------------------+---------------+--------------------+
|https://www.zomat...|942, 21st Main Ro...|         Jalsa|         

In [30]:
df.na.fill('not_null').show(3)
# approx_cost(for two people)과 votes는 IntegerType이므로 string인 'not_null'로 대체할 수 없으므로
# 값이 NULL이라도 대체되지 않는다.
# fill(str)이라면 data type이 string인 열의 값만 대체된다.

+--------------------+--------------------+--------------+--------------------+--------------------+-----+-----+-------------+--------------------+-------------+--------------------+--------------------+---------------------------+--------------------+--------------------+---------------+--------------------+
|                 url|             address|          name|        online_order|          book_table| rate|votes|        phone|            location|    rest_type|          dish_liked|            cuisines|approx_cost(for two people)|        reviews_list|           menu_item|listed_in(type)|     listed_in(city)|
+--------------------+--------------------+--------------+--------------------+--------------------+-----+-----+-------------+--------------------+-------------+--------------------+--------------------+---------------------------+--------------------+--------------------+---------------+--------------------+
|https://www.zomat...|942, 21st Main Ro...|         Jalsa|         

#### **1)** data type이 numeric value(`int, float` 등)인 경우, `NULL`값을 **해당 column의 평균값**으로 대체할 수 있음

- `votes` column의 NULL값을 `votes` column의 **평균값으로 대체**

In [31]:
from pyspark.sql.functions import *

In [32]:
df.agg(mean(df.votes)).show()
# df.agg(avg(df.votes)).show()와 동일

+------------------+
|        avg(votes)|
+------------------+
|283.71422493811883|
+------------------+



In [34]:
# df.agg(mean(df.votes))가 반환하는 결과는 DataFrame 타입으로
# na.fill의 인자로 사용할 수 없음
# fill의 인자는 int, float, string, bool or dict만 가능
# vote열의 평균값(float)을 avg_votes라는 변수에 저장
avg_votes = df.agg(mean(df.votes)).collect()[0][0]
# df.agg(mean(df.votes)).collect(): DataFrame의 각 Row객체를 list로 반환
# df.agg(mean(df.votes)).collect()[0]: DataFrame의 첫번째 행(Row)
# df.agg(mean(df.votes)).collect()[0][0]: DataFrame의 첫번째 행(Row), 첫번째 요소
print(avg_votes)
print(type(avg_votes))

283.71422493811883
<class 'float'>


In [35]:
avg_votes

283.71422493811883

- `DataFrame.na.fill(val,subset)`: `subset` 열의 값이 `NULL`인 경우, `val`로 대체함

In [36]:
# NULL을 채워넣기 전, NULL의 수
df.filter(df.votes.isNull()).count()

20018

In [37]:
# votes의 평균값으로 NULL을 대체
df_votes_fill = df.na.fill(avg_votes,subset='votes')
# 위 과정 이후, votes 열이 NULL인 행의 수
df_votes_fill.filter(df_votes_fill.votes.isNull()).count()

0

- 실제 평균값은 `float` 타입으로 `283.71422493811883`이었으나, `votes`열의 data type이 `IntegerType`이므로 `283`으로 대체됨

In [38]:
print(df.collect()[10]['votes'])
print(df_votes_fill.collect()[10]['votes'])

None
283


#### **2)** Data Type이 `str`인 **categorical data**(한정적인 종류의 값을 가진 데이터 ex> 남/여, 봄/여름/가을/겨울)의 경우, **가장 많이 등장한 값으로 대체**할 수 있음

- `mode(col)`: column에서 가장 많이 등장한 값 반환

In [39]:
df.agg(mode(df['listed_in(type)'])).show()

+---------------------+
|mode(listed_in(type))|
+---------------------+
|             Delivery|
+---------------------+



In [43]:
type_freq = df.agg(mode(df['listed_in(type)'])).collect()[0][0]

In [45]:
df.select('listed_in(type)').show(10)

+--------------------+
|     listed_in(type)|
+--------------------+
|                NULL|
|        ('Rated 4.0'|
|        ('Rated 5.0'|
| ""RATED\n \nWent...|
| 'RATED\n  Reache...|
|                NULL|
|                NULL|
|                NULL|
|                NULL|
|                NULL|
+--------------------+
only showing top 10 rows



In [46]:
df_type_fill = df.na.fill(type_freq,subset='listed_in(type)')
df_type_fill.select('listed_in(type)').show(10)

+--------------------+
|     listed_in(type)|
+--------------------+
|            Delivery|
|        ('Rated 4.0'|
|        ('Rated 5.0'|
| ""RATED\n \nWent...|
| 'RATED\n  Reache...|
|            Delivery|
|            Delivery|
|            Delivery|
|            Delivery|
|            Delivery|
+--------------------+
only showing top 10 rows

